# Churn Label Definition

This notebook defines the target variable for customer churn prediction.

The churn label is constructed from customer ordering behaviour using the Instacart evaluation structure while avoiding the use of future behaviour as predictive features.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")
FEATURE_DIR = Path("../data/interim")

features = pd.read_csv(
    FEATURE_DIR / "customer_features.csv"
)

orders = pd.read_csv(
    DATA_DIR / "orders.csv"
)

print("Features:", features.shape)
print("Orders:", orders.shape)

Features: (206209, 24)
Orders: (3421083, 7)


In [3]:
orders["eval_set"].value_counts()

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

In [4]:
orders.groupby("eval_set")["user_id"].nunique()

eval_set
prior    206209
test      75000
train    131209
Name: user_id, dtype: int64

## Customer Evaluation Structure

The Instacart dataset separates customer orders into historical prior orders and a held-out train/test evaluation structure.

The evaluation structure is inspected before defining the churn target so that future customer behaviour is not accidentally included in predictive features.

In [5]:
orders[
    [
        "user_id",
        "order_number",
        "eval_set"
    ]
].sort_values(
    ["user_id", "order_number"]
).head(20)

,user_id,order_number,eval_set
0,1,1,prior
1,1,2,prior
2,1,3,prior
3,1,4,prior
4,1,5,prior
5,1,6,prior
6,1,7,prior
7,1,8,prior
8,1,9,prior
9,1,10,prior


In [6]:
historical_intervals = (
    orders[orders["eval_set"] == "prior"]
    .groupby("user_id")["days_since_prior_order"]
    .median()
    .rename("historical_median_interval")
)

future_orders = (
    orders[orders["eval_set"].isin(["train", "test"])]
    [
        [
            "user_id",
            "eval_set",
            "order_number",
            "days_since_prior_order"
        ]
    ]
    .rename(
        columns={
            "days_since_prior_order": "future_order_gap"
        }
    )
)

label_analysis = future_orders.merge(
    historical_intervals,
    on="user_id",
    how="left"
)

label_analysis["future_gap_ratio"] = (
    label_analysis["future_order_gap"]
    / label_analysis["historical_median_interval"].replace(0, pd.NA)
)

label_analysis["future_gap_ratio"].describe()

count     206103.0
unique      1085.0
top            1.0
freq       23967.0
Name: future_gap_ratio, dtype: float64

In [7]:
label_analysis[
    ["future_order_gap", "historical_median_interval", "future_gap_ratio"]
].describe()

,future_order_gap,historical_median_interval
count,206209.000000,206209.000000
mean,17.061782,14.753522
std,10.672178,8.676030
min,0.000000,0.000000
25%,7.000000,7.000000
50%,15.000000,13.000000
75%,30.000000,21.500000
max,30.000000,30.000000


## Customer-Relative Churn Signal

Churn behaviour is evaluated relative to each customer's historical purchasing cycle.

The future order interval is compared with the customer's historical median purchase interval. This avoids imposing a universal calendar threshold on a dataset that does not provide absolute order dates.

The resulting ratio represents how much longer the customer's next observed purchase took compared with their typical historical cycle.

In [8]:
future_orders = (
    orders[orders["eval_set"].isin(["train", "test"])]
    [
        [
            "user_id",
            "eval_set",
            "order_number",
            "days_since_prior_order",
        ]
    ]
    .rename(
        columns={
            "days_since_prior_order": "future_order_gap"
        }
    )
)

historical_cycle = (
    orders[orders["eval_set"] == "prior"]
    .groupby("user_id")["days_since_prior_order"]
    .median()
    .rename("historical_median_interval")
)

label_data = future_orders.merge(
    historical_cycle,
    on="user_id",
    how="left",
    validate="one_to_one",
)

label_data["future_gap_ratio"] = (
    label_data["future_order_gap"]
    / label_data["historical_median_interval"].replace(0, pd.NA)
)

label_data.shape

(206209, 6)

In [9]:
label_data["future_gap_ratio"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count     206103.0
unique      1085.0
top            1.0
freq       23967.0
Name: future_gap_ratio, dtype: float64

In [10]:
thresholds = [1.25, 1.5, 1.75, 2.0, 2.5, 3.0]

threshold_analysis = []

for threshold in thresholds:
    valid_ratio = label_data["future_gap_ratio"].notna()

    churn_count = (
        (label_data["future_gap_ratio"] > threshold)
        & valid_ratio
    ).sum()

    total_valid = valid_ratio.sum()

    threshold_analysis.append({
        "threshold": threshold,
        "churn_count": int(churn_count),
        "non_churn_count": int(total_valid - churn_count),
        "churn_rate": float(churn_count / total_valid),
    })

pd.DataFrame(threshold_analysis)

,threshold,churn_count,non_churn_count,churn_rate
0,1.25,82200,123903,0.398830
1,1.50,62576,143527,0.303615
2,1.75,50538,155565,0.245207
3,2.00,38263,167840,0.185650
4,2.50,25154,180949,0.122046
5,3.00,17047,189056,0.082711


## Handling Zero Historical Median Intervals

A small number of customers have a historical median purchase interval of zero because they made multiple orders with zero days between consecutive purchases.

For these customers, the historical mean purchase interval is used as a fallback reference when it is greater than zero.

This avoids division by zero while preserving the customer's observed purchasing behaviour.

In [11]:
historical_stats = (
    orders[orders["eval_set"] == "prior"]
    .groupby("user_id")["days_since_prior_order"]
    .agg(
        historical_median_interval="median",
        historical_mean_interval="mean"
    )
    .reset_index()
)

label_data = (
    future_orders
    .drop(columns=["historical_median_interval"], errors="ignore")
    .merge(
        historical_stats,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

label_data["reference_interval"] = (
    label_data["historical_median_interval"]
)

zero_median_mask = (
    label_data["reference_interval"].eq(0)
)

label_data.loc[
    zero_median_mask,
    "reference_interval"
] = label_data.loc[
    zero_median_mask,
    "historical_mean_interval"
]

label_data["future_gap_ratio"] = (
    label_data["future_order_gap"]
    / label_data["reference_interval"].replace(0, pd.NA)
)

print("Total customers:", len(label_data))
print(
    "Zero reference intervals:",
    label_data["reference_interval"].eq(0).sum()
)
print(
    "Missing future gap ratios:",
    label_data["future_gap_ratio"].isna().sum()
)

Total customers: 206209
Zero reference intervals: 24
Missing future gap ratios: 24


In [12]:
label_data["churn_like_label"] = (
    label_data["future_gap_ratio"] >= 2.0
).astype("int8")

In [13]:
label_distribution = (
    label_data["churn_like_label"]
    .value_counts()
    .sort_index()
    .rename_axis("churn_like_label")
    .reset_index(name="customer_count")
)

label_distribution["percentage"] = (
    label_distribution["customer_count"]
    / len(label_data)
    * 100
)

label_distribution

,churn_like_label,customer_count,percentage
0,0,163048,79.069294
1,1,43161,20.930706


In [14]:
label_data[
    [
        "user_id",
        "future_order_gap",
        "reference_interval",
        "future_gap_ratio",
        "churn_like_label"
    ]
].head(20)

,user_id,future_order_gap,reference_interval,future_gap_ratio,churn_like_label
0,1,14.0,20.0,0.7,0
1,2,30.0,13.0,2.307692,1
2,3,11.0,11.0,1.0,0
3,4,30.0,17.0,1.764706,0
4,5,6.0,11.0,0.545455,0
5,6,22.0,9.0,2.444444,1
6,7,6.0,7.0,0.857143,0
7,8,10.0,30.0,0.333333,0
8,9,30.0,18.0,1.666667,0
9,10,30.0,18.5,1.621622,0


In [15]:
zero_reference_customers = label_data[
    label_data["reference_interval"].eq(0)
][
    [
        "user_id",
        "future_order_gap",
        "historical_median_interval",
        "historical_mean_interval",
        "reference_interval",
    ]
]

zero_reference_customers

,user_id,future_order_gap,historical_median_interval,historical_mean_interval,reference_interval
9514,9515,13.0,0.0,0.0,0.0
14432,14433,0.0,0.0,0.0,0.0
15494,15495,4.0,0.0,0.0,0.0
36903,36904,12.0,0.0,0.0,0.0
58933,58934,13.0,0.0,0.0,0.0
62179,62180,3.0,0.0,0.0,0.0
64974,64975,8.0,0.0,0.0,0.0
74328,74329,0.0,0.0,0.0,0.0
74659,74660,30.0,0.0,0.0,0.0
80566,80567,0.0,0.0,0.0,0.0


In [16]:
zero_users = zero_reference_customers["user_id"].tolist()

orders[
    (orders["eval_set"] == "prior")
    & (orders["user_id"].isin(zero_users))
][
    [
        "user_id",
        "order_number",
        "days_since_prior_order"
    ]
].sort_values(
    ["user_id", "order_number"]
)

,user_id,order_number,days_since_prior_order
158393,9515,1,NaN
158394,9515,2,0.0
158395,9515,3,0.0
238094,14433,1,NaN
238095,14433,2,0.0
...,...,...,...
3007649,181478,2,0.0
3007650,181478,3,0.0
3339763,201321,1,NaN
3339764,201321,2,0.0


## Final Churn Label Definition

A customer is classified as churn-like when their next observed purchase occurs at least twice as long as their historical purchase-cycle reference interval.

Customers with no measurable historical purchase-cycle interval are excluded from supervised label construction because a relative delay cannot be calculated reliably for them.

In [17]:
valid_label_mask = (
    label_data["reference_interval"] > 0
)

label_data["churn_like_label"] = pd.NA

label_data.loc[
    valid_label_mask,
    "churn_like_label"
] = (
    label_data.loc[
        valid_label_mask,
        "future_gap_ratio"
    ] >= 2.0
).astype("int8")

In [18]:
print(
    "Total customers:",
    len(label_data)
)

print(
    "Valid labelled customers:",
    label_data["churn_like_label"].notna().sum()
)

print(
    "Excluded customers:",
    label_data["churn_like_label"].isna().sum()
)

Total customers: 206209
Valid labelled customers: 206185
Excluded customers: 24


In [19]:
label_distribution = (
    label_data[
        label_data["churn_like_label"].notna()
    ]
    ["churn_like_label"]
    .value_counts()
    .sort_index()
)

label_distribution

churn_like_label
0    163024
1     43161
Name: count, dtype: int64

In [20]:
final_labels = (
    label_data[
        [
            "user_id",
            "churn_like_label"
        ]
    ]
    .dropna(subset=["churn_like_label"])
    .copy()
)

final_labels["churn_like_label"] = (
    final_labels["churn_like_label"]
    .astype("int8")
)

label_output_path = (
    Path("../data/interim")
    / "churn_labels.csv"
)

final_labels.to_csv(
    label_output_path,
    index=False
)

print(f"Saved: {label_output_path}")
print("Shape:", final_labels.shape)
print(
    "Duplicate user_id:",
    final_labels["user_id"].duplicated().sum()
)

Saved: ..\data\interim\churn_labels.csv
Shape: (206185, 2)
Duplicate user_id: 0
